# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Note: dataset.metadata is an object; access with dot notation
print(f"{dataset.metadata.name}: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their @id
print("Available record sets in this dataset:")
for record_set in dataset.record_sets:
    print(f"  @id: {record_set.id} | name: {record_set.name}")

# For each record set, print fields and columns
print("\nRecord set details:")
for record_set in dataset.record_sets:
    print(f"\nRecord set @id: {record_set.id} | name: {record_set.name}")
    for field in record_set.fields:
        col_ids = [col.id for col in field.columns]
        print(f"    Field @id: {field.id}, name: {field.name}, dataType: {field.data_type}, column(s) @id: {col_ids}")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect all record set @id values
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load records as DataFrame
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set: {record_set_id}")

# Example: show columns from the main record set
# Pick the first record_set by default for display
example_record_set_id = record_set_ids[0] if record_set_ids else None
if example_record_set_id:
    print('\nColumns for record set:', example_record_set_id)
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())
else:
    print("No record sets found in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
# Select one record set and numeric field for demo EDA
# We'll try to automatically select a numeric column if present
import numpy as np

chosen_record_set_id = example_record_set_id
df = dataframes[chosen_record_set_id]

# Attempt to auto-detect numeric fields (float or int columns)
potential_numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
if not potential_numeric_columns:
    # Try to parse columns with float-looking contents
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            pass
    potential_numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()

if potential_numeric_columns:
    numeric_field = potential_numeric_columns[0]
    print(f"Using numeric field: {numeric_field}")
else:
    print("No numeric columns found for EDA.")
    numeric_field = None

if numeric_field:
    threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Pick a group-by field if present
    group_field = None
    for col in df.columns:
        if col != numeric_field and df[col].nunique() < min(10, len(df)//2):
            group_field = col
            break

    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean of {numeric_field} by {group_field}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Simple histogram and scatterplot for the numeric field, if available
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and not df.empty:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.xlabel(numeric_field)
    plt.title(f'Distribution of {numeric_field}')
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field found for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we successfully loaded the FAIR² dataset: "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" using the `mlcroissant` library. We previewed metadata, record sets, and fields using their Croissant `@id`s. Basic filtering, normalization, grouping, and visualization steps provided initial insights into the protein-level mass spectrometry data. For deeper studies, consult the dataset documentation and the full Croissant schema for field definitions and advanced analytic opportunities.